# L2: Multi-agent Customer Support Automation

In this lesson, you will learn about the six key elements which help make Agents perform even better:
- Role Playing
- Focus
- Tools
- Cooperation
- Guardrails
- Memory

In [13]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import libraries, API and LLM

In [14]:
from crewai import Agent, Task, Crew

In [15]:
import os
from utils import get_openai_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-4o-mini'

## Role Playing, Focus and Cooperation

In [16]:
support_agent = Agent(
    role="Senior Support Representative",
	goal="Be the most friendly and helpful "
        "support representative in your team",
	backstory=(
		"You work at crewAI (https://crewai.com) and "
        " are now working on providing "
		"support to {customer}, a super important customer "
        " for your company."
		"You need to make sure that you provide the best support!"
		"Make sure to provide full complete answers, "
        " and make no assumptions."
	),
	allow_delegation=False,
	verbose=True
)

- By not setting `allow_delegation=False`, `allow_delegation` takes its default value of being `True`.
- This means the agent _can_ delegate its work to another agent which is better suited to do a particular task. 

In [17]:
support_quality_assurance_agent = Agent(
	role="Support Quality Assurance Specialist",
	goal="Get recognition for providing the "
    "best support quality assurance in your team",
	backstory=(
		"You work at crewAI (https://crewai.com) and "
        "are now working with your team "
		"on a request from {customer} ensuring that "
        "the support representative is "
		"providing the best support possible.\n"
		"You need to make sure that the support representative "
        "is providing full"
		"complete answers, and make no assumptions."
	),
	verbose=True
)

* **Role Playing**: Both agents have been given a role, goal and backstory.
* **Focus**: Both agents have been prompted to get into the character of the roles they are playing.
* **Cooperation**: Support Quality Assurance Agent can delegate work back to the Support Agent, allowing for these agents to work together.

## Tools, Guardrails and Memory

### Tools

- Import CrewAI tools

In [ ]:
from crewai_tools import ScrapeWebsiteTool

### Possible Custom Tools
- Load customer data
- Tap into previous conversations
- Load data from a CRM
- Checking existing bug reports
- Checking existing feature requests
- Checking ongoing tickets
- ... and more

- Some ways of using CrewAI tools.

```Python
search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()
```

- Instantiate a document scraper tool.
- The tool will scrape a page (only 1 URL) of the CrewAI documentation.

In [19]:
docs_scrape_tool = ScrapeWebsiteTool(
    website_url="https://docs.crewai.com/how-to/Creating-a-Crew-and-kick-it-off/"
)

##### Different Ways to Give Agents Tools

- Agent Level: The Agent can use the Tool(s) on any Task it performs.
- Task Level: The Agent will only use the Tool(s) when performing that specific Task.

**Note**: Task Tools override the Agent Tools.

### Creating Tasks
- You are passing the Tool on the Task Level.

In [20]:
inquiry_resolution = Task(
    description=(
        "{customer} just reached out with a super important ask:\n"
	    "{inquiry}\n\n"
        "{person} from {customer} is the one that reached out. "
		"Make sure to use everything you know "
        "to provide the best support possible."
		"You must strive to provide a complete "
        "and accurate response to the customer's inquiry."
    ),
    expected_output=(
	    "A detailed, informative response to the "
        "customer's inquiry that addresses "
        "all aspects of their question.\n"
        "The response should include references "
        "to everything you used to find the answer, "
        "including external data or solutions. "
        "Ensure the answer is complete, "
		"leaving no questions unanswered, and maintain a helpful and friendly "
		"tone throughout."
    ),
	tools=[docs_scrape_tool],
    agent=support_agent,
)

- `quality_assurance_review` is not using any Tool(s)
- Here the QA Agent will only review the work of the Support Agent

In [21]:
quality_assurance_review = Task(
    description=(
        "Review the response drafted by the Senior Support Representative for {customer}'s inquiry. "
        "Ensure that the answer is comprehensive, accurate, and adheres to the "
		"high-quality standards expected for customer support.\n"
        "Verify that all parts of the customer's inquiry "
        "have been addressed "
		"thoroughly, with a helpful and friendly tone.\n"
        "Check for references and sources used to "
        " find the information, "
		"ensuring the response is well-supported and "
        "leaves no questions unanswered."
    ),
    expected_output=(
        "A final, detailed, and informative response "
        "ready to be sent to the customer.\n"
        "This response should fully address the "
        "customer's inquiry, incorporating all "
		"relevant feedback and improvements.\n"
		"Don't be too formal, we are a chill and cool company "
	    "but maintain a professional and friendly tone throughout."
    ),
    agent=support_quality_assurance_agent,
)


### Creating the Crew

#### Memory
- Setting `memory=True` when putting the crew together enables Memory.

In [22]:
crew = Crew(
  agents=[support_agent, support_quality_assurance_agent],
  tasks=[inquiry_resolution, quality_assurance_review],
  verbose=True,
  memory=True
)

### Running the Crew

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

#### Guardrails
- By running the execution below, you can see that the agents and the responses are within the scope of what we expect from them.

In [23]:
inputs = {
    "customer": "DeepLearningAI",
    "person": "Andrew Ng",
    "inquiry": "I need help with setting up a Crew "
               "and kicking it off, specifically "
               "how can I add memory to my crew? "
               "Can you provide guidance?"
}
result = crew.kickoff(inputs=inputs)

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 92c1103b-22de-4c27-b37f-760fe94d1000                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭────────────────────────────────────────────── 🧠 Retrieved Memory ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Recent Insights:                                                                                               │
│  - Final Answer:                                                                                                │
│                                                                                                                 │
│  To set up a Crew in crewAI and add memory to it, you can follow the steps outlined in the documentation        │
│  provided. Below is a detailed guide based on the information retrieved from the CrewAI documentation:          │
│                                                                                                                 │
│  1. Start by understanding the overview of CrewAI concepts, architecture, and what you can build with agents,   │
│  crews, and flows.                                                                                              │
│  2. Install CrewAI via uv, configure API keys, and set up the CLI for local development.                        │
│  3. Utilize the quickstart guide to spin ...                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────── Retrieval Time: 937.09ms ────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Support Representative                                                                           │
│                                                                                                                 │
│  Task: DeepLearningAI just reached out with a super important ask:                                              │
│  I need help with setting up a Crew and kicking it off, specifically how can I add memory to my crew? Can you   │
│  provide guidance?                                                                                              │
│                                                                                                                 │
│  Andrew Ng from DeepLearningAI is the one that reached out. Make sure to use everything you know to provide     │
│  the best support possible.You must strive to provide a complete and accurate response to the customer's        │
│  inquiry.                                                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Support Quality Assurance Specialist                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  To set up a Crew in crewAI and add memory to it, you can follow the steps outlined below, which provide a      │
│  comprehensive guide based on the information retrieved from the CrewAI documentation:                          │
│                                                                                                                 │
│  1. **Overview**: Familiarize yourself with CrewAI's core concepts including agents, crews, and flows.          │
│  Understanding these elements is crucial, as memory is a key feature that allows agents to remember             │
│  information across multiple interactions and sessions, enhancing their ability to assist users effectively.    │
│                                                                                                                 │
│  2. **Setting Up**: Begin by installing CrewAI via `uv`. Make sure to configure your API keys correctly and     │
│  set up the Command Line Interface (CLI) to facilitate local development. This foundational step is essential   │
│  to ensure you have a functional environment to work in.                                                        │
│                                                                                                                 │
│  3. **Creating Your Crew**: Utilize the quickstart guide available in the documentation to spin up your first   │
│  crew quickly. This will help you get acquainted with the platform's core runtime, project layout, and          │
│  development loop, allowing you to start building effectively.                                                  │
│                                                                                                                 │
│  4. **Composing Agents**: When creating agents, you can integrate memory to enable them to store and recall     │
│  information. This is done using structured outputs defined with Pydantic, where you can specify the            │
│  functionality of the memory feature. Be mindful of the types of memory: for instance, short-term memory for    │
│  quick interactions or long-term memory for data that persists across sessions.                                 │
│                                                                                                                 │
│  5. **Implementation of Memory**: As you’re composing your agents, take care to decide exactly what data your   │
│  agent needs to retain. Consider how this memory will be accessed and updated during interactions, ensuring     │
│  your agent performs optimally according to user expectations.                                                  │
│                                                                                                                 │
│  6. **Testing**: Once you've added memory to your agents, it’s vital to test them thoroughly to confirm they    │
│  are correctly retaining and recalling information as intended. You can conduct tests during the orchestration  │
│  of your flows where your agents are actively engaged to validate their functionality.                          │
│                                                                                                                 │
│  7. **Review Documentation**: For more detailed step-by-step instructions and code examples on integrating      │
│  memory with your agents, please reference the CrewAI d

Output()

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 5c59d697-8447-4895-a573-67a7bfb50c74                                                                     │
│  Agent: Support Quality Assurance Specialist                                                                    │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 92c1103b-22de-4c27-b37f-760fe94d1000                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: To set up a Crew in crewAI and add memory to it, you can follow the steps outlined below, which  │
│  provide a comprehensive guide based on the information retrieved from the CrewAI documentation:                │
│                                                                                                                 │
│  1. **Overview**: Familiarize yourself with CrewAI's core concepts including agents, crews, and flows.          │
│  Understanding these elements is crucial, as memory is a key feature that allows agents to remember             │
│  information across multiple interactions and sessions, enhancing their ability to assist users effectively.    │
│                                                                                                                 │
│  2. **Setting Up**: Begin by installing CrewAI via `uv`. Make sure to configure your API keys correctly and     │
│  set up the Command Line Interface (CLI) to facilitate local development. This foundational step is essential   │
│  to ensure you have a functional environment to work in.                                                        │
│                                                                                                                 │
│  3. **Creating Your Crew**: Utilize the quickstart guide available in the documentation to spin up your first   │
│  crew quickly. This will help you get acquainted with the platform's core runtime, project layout, and          │
│  development loop, allowing you to start building effectively.                                                  │
│                                                                                                                 │
│  4. **Composing Agents**: When creating agents, you can integrate memory to enable them to store and recall     │
│  information. This is done using structured outputs defined with Pydantic, where you can specify the            │
│  functionality of the memory feature. Be mindful of the types of memory: for instance, short-term memory for    │
│  quick interactions or long-term memory for data that persists across sessions.                                 │
│                                                                                                                 │
│  5. **Implementation of Memory**: As you’re composing your agents, take care to decide exactly what data your   │
│  agent needs to retain. Consider how this memory will be accessed and updated during interactions, ensuring     │
│  your agent performs optimally according to user expectations.                                                  │
│                                                                                                                 │
│  6. **Testing**: Once you've added memory to your agents, it’s vital to test them thoroughly to confirm they    │
│  are correctly retaining and recalling information as intended. You can conduct tests during the orchestration  │
│  of your flows where your agents are actively engaged to validate their functionality.                          │
│                                                                                                                 │
│  7. **Review Documentation**: For more detailed step-b

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

- Display the final result as Markdown.

In [24]:
from IPython.display import Markdown
Markdown(result.raw)

To set up a Crew in crewAI and add memory to it, you can follow the steps outlined below, which provide a comprehensive guide based on the information retrieved from the CrewAI documentation:

1. **Overview**: Familiarize yourself with CrewAI's core concepts including agents, crews, and flows. Understanding these elements is crucial, as memory is a key feature that allows agents to remember information across multiple interactions and sessions, enhancing their ability to assist users effectively.

2. **Setting Up**: Begin by installing CrewAI via `uv`. Make sure to configure your API keys correctly and set up the Command Line Interface (CLI) to facilitate local development. This foundational step is essential to ensure you have a functional environment to work in.

3. **Creating Your Crew**: Utilize the quickstart guide available in the documentation to spin up your first crew quickly. This will help you get acquainted with the platform's core runtime, project layout, and development loop, allowing you to start building effectively.

4. **Composing Agents**: When creating agents, you can integrate memory to enable them to store and recall information. This is done using structured outputs defined with Pydantic, where you can specify the functionality of the memory feature. Be mindful of the types of memory: for instance, short-term memory for quick interactions or long-term memory for data that persists across sessions.

5. **Implementation of Memory**: As you’re composing your agents, take care to decide exactly what data your agent needs to retain. Consider how this memory will be accessed and updated during interactions, ensuring your agent performs optimally according to user expectations.

6. **Testing**: Once you've added memory to your agents, it’s vital to test them thoroughly to confirm they are correctly retaining and recalling information as intended. You can conduct tests during the orchestration of your flows where your agents are actively engaged to validate their functionality.

7. **Review Documentation**: For more detailed step-by-step instructions and code examples on integrating memory with your agents, please reference the CrewAI documentation available at [CrewAI Documentation](https://docs.crewai.com/how-to/Creating-a-Crew-and-kick-it-off/).

If you have further specific requirements or additional questions regarding setting up memory in your crew, feel free to reach out. We're always here to help you succeed and ensure your experience with CrewAI is smooth and productive! 

You can look forward to enabling more dynamic interactions for your agents through effective memory management!